### Data Ingestion

In [2]:
import pandas as pd
from typing import List, Dict, Any
import os

In [3]:
from langchain_core.documents import Document ## Data structure for langchain
from langchain_text_splitters import ( ## importing the text splitters
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

c:\Users\kumar\OneDrive\Desktop\Agentic_AI\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Document structure in Langchain

In [4]:
## Creating simple document
document = Document(
    page_content="Test data on langchain",
    metadata = { ## provides information on the document when it is searched through vector data base
        "source" : "Test data through codespaces",
        "page" : 1,
        "date_created" : "15-07-2026",
        "document_structure" : "Test data"
    }
)

print(f"Page Content : {document.page_content}")
print(f"MetaData : {document.metadata}")

Page Content : Test data on langchain
MetaData : {'source': 'Test data through codespaces', 'page': 1, 'date_created': '15-07-2026', 'document_structure': 'Test data'}


In [5]:
type(document)

langchain_core.documents.base.Document

### Reading Text Files

In [6]:
## Create file a simple text file
import os
os.makedirs("data/text_files", exist_ok=True)

In [7]:
sample_text_file = {
    "data/text_files/purview_intro.txt":"""

Microsoft Purview Records Management will integrate with Power Automate to allow administrators to run custom workflows when retention-labeled items reach the end of their retention period. This enables automated post-retention handling using Power Automate across Microsoft 365 workloads.

This message is associated with Roadmap ID 558859.

Rollout Schedule:

General Availability (GCC, GCCH, and DoD): Rollout begins in January 2027 (previously mid-July 2026) and is expected to complete by end of January 2027 (previously late September 2026).

Impact on Your Organization:

Who is affected: Records Management and Compliance administrators

""",
"data/text_files/bifurcation.txt": """

Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message content, but different envelopes.

Bifurcation occurs through Microsoft Exchange while messages are in transit.

Why bifurcation?
There are different purposes for which bifurcation can occur to a message in transit, such as (including but not limited) recipient-based customization, routing, security, and performance.

"""

}

for filepath, content in sample_text_file.items():
    with open(filepath,"w", encoding="utf-8") as f:
        f.write(content)

print("File created")

File created


### TextLoader - Can read Single File

In [8]:
from langchain_community.document_loaders import TextLoader ## TextLoader will automactically load any single file to langchain data structure document data structure

## loading a single text file

loader = TextLoader("data/text_files/purview_intro.txt", encoding="utf-8")

documents = loader.load()

print(type(documents))
print(documents)
print(f"Content Preview :{documents[0].page_content[:177]}...")
print(f"Metadata: {documents[0].metadata}")



<class 'list'>
[Document(metadata={'source': 'data/text_files/purview_intro.txt'}, page_content='\n\nMicrosoft Purview Records Management will integrate with Power Automate to allow administrators to run custom workflows when retention-labeled items reach the end of their retention period. This enables automated post-retention handling using Power Automate across Microsoft 365 workloads.\n\nThis message is associated with Roadmap ID 558859.\n\nRollout Schedule:\n\nGeneral Availability (GCC, GCCH, and DoD): Rollout begins in January 2027 (previously mid-July 2026) and is expected to complete by end of January 2027 (previously late September 2026).\n\nImpact on Your Organization:\n\nWho is affected: Records Management and Compliance administrators\n\n')]
Content Preview :

Microsoft Purview Records Management will integrate with Power Automate to allow administrators to run custom workflows when retention-labeled items reach the end of their ret...
Metadata: {'source': 'data/text_files/p

C:\Users\kumar\AppData\Local\Temp\ipykernel_5112\2518219537.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader ## TextLoader will automactically load any single file to langchain data structure document data structure


### DirectoryLoader - Can able to read Multiple files

In [9]:
from langchain_community.document_loaders import DirectoryLoader

## Load all the files in the data folder with the help of DirectoryLoader module

dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## regular expression to match files
    loader_cls= TextLoader, ## loader class to use
    loader_kwargs= {'encoding': 'utf-8'},
    show_progress= True
)

documents = dir_loader.load()
print(documents)

# enumerating documents
for i, doc in enumerate(documents):
    print(f"\n Document number - {i+1}")
    print(f"Source of the Document : {doc.metadata['source']}")
    print(f"Length of each document - {doc.metadata['source']} is {len(doc.page_content)}")

# Disadvantages of DirectoryLoader

# - All the files should be same type
# - Limited error handling per file
# - Can be memmory intensive for large directories


# Format of the document structure - 

# Plaintext
# List [
#   └── Document Object {
#         ├── .page_content ➔ (String)
#         └── .metadata     ➔ Dictionary {
#                                └── 'source' ➔ (String)
#                             }
#       }
# ]

# Instead of a plain dictionary, LangChain wraps this data inside a custom class (class Document). This means you access the data using dot notation (.) for the main attributes, and standard bracket notation ([]) for the metadata dictionary:
    
# Data     TypeSyntax                          What it represents

# List	documents[0]	Gets the first document object in the list.

# Object Property	doc.page_content	Standard string property containing the raw text.

# Object Property	doc.metadata	This specific property is a standard Python dictionary.

# Dictionary Key	doc.metadata['source']	Standard dictionary lookup to grab the string file path.


100%|██████████| 2/2 [00:00<00:00, 118.12it/s]

[Document(metadata={'source': 'data\\text_files\\bifurcation.txt'}, page_content='\n\nBifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message content, but different envelopes.\n\nBifurcation occurs through Microsoft Exchange while messages are in transit.\n\nWhy bifurcation?\nThere are different purposes for which bifurcation can occur to a message in transit, such as (including but not limited) recipient-based customization, routing, security, and performance.\n\n'), Document(metadata={'source': 'data\\text_files\\purview_intro.txt'}, page_content='\n\nMicrosoft Purview Records Management will integrate with Power Automate to allow administrators to run custom workflows when retention-labeled items reach the end of their retention period. This enables automated post-retention handling using Power Automate across Microsoft 365 workloads.\n\nThis message is associated with Roadmap ID 558859.\n\

### Text Splitting Strategies 

In [10]:
from langchain_text_splitters import ( ## importing the text splitters that split the documents into smaller chunks
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)


In [11]:
### Method - 1 Character text splitter 

text = documents[0].page_content

char_splitter = CharacterTextSplitter(
    separator="\n", # split on new lines
    chunk_size = 200, # max chunk size in characters
    chunk_overlap = 20, # overlapping between chunks (continuation of characters in each chunk for preserving semantic meaning of the information)
    length_function = len # how to measure chunk size
)

In [12]:
char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}")

Created 3 chunks
First chunk: Bifurcation (also known as forking) refers to the process of creating multiple copies of a given mes


In [13]:
for i in range(len(char_chunks)):
    print(f"Length of {i+1} chunk is {len(char_chunks[i])}")
    print(f"Information in the {i+1} chunk is  - {char_chunks[i]}")

Length of 1 chunk is 183
Information in the 1 chunk is  - Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message content, but different envelopes.
Length of 2 chunk is 93
Information in the 2 chunk is  - Bifurcation occurs through Microsoft Exchange while messages are in transit.
Why bifurcation?
Length of 3 chunk is 188
Information in the 3 chunk is  - There are different purposes for which bifurcation can occur to a message in transit, such as (including but not limited) recipient-based customization, routing, security, and performance.


In [14]:
## Method 2: Recursive character splitting (RECOMMENDED)

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"," ","\n"], ## This text splitter involves having multiple seperators
    chunk_size = 200, # max chunk size in characters
    chunk_overlap = 20, # overlapping between chunks (continuation of characters in each chunk for preserving semantic meaning of the information)
    length_function = len # how to measure chunk size
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}")

for i in range(len(recursive_chunks)):
    print(f"Length of {i+1} chunk is {len(recursive_chunks[i])}")
    print(f"Information in the {i+1} chunk is  - {recursive_chunks[i]}")

Created 4 chunks
First chunk: Bifurcation (also known as forking) refers to the process of creating multiple copies of a given mes
Length of 1 chunk is 183
Information in the 1 chunk is  - Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message content, but different envelopes.
Length of 2 chunk is 76
Information in the 2 chunk is  - Bifurcation occurs through Microsoft Exchange while messages are in transit.
Length of 3 chunk is 192
Information in the 3 chunk is  - Why bifurcation?
There are different purposes for which bifurcation can occur to a message in transit, such as (including but not limited) recipient-based customization, routing, security, and
Length of 4 chunk is 26
Information in the 4 chunk is  - security, and performance.


### Token Based Spillting

In [ ]:
token_splitter = TokenTextSplitter(
    chunk_size = 50, # max chunk size in tokens (not characters) tokens can be spaces which is called tokenization (eg - ["Understanding", " token", "ization", " is", " vital", "."])
    chunk_overlap = 20, # overlapping between chunks (continuation of characters in each chunk for preserving semantic meaning of the information)
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}")

Created 3 chunks
First chunk: 

Bifurcation (also known as forking) refers to the process of creating multiple copies of a given m


: 